# XYZ-Scan Viewer

Lädt Scanner-`.XYZ`-Dateien und zeigt sie interaktiv (drehbar) plus Projektionen und Z-Histogramm.

Das Scanner-Format hat Kopfzeilen (`L####…`, `##`-getrennt) und Datenzeilen `X Y Z nx ny nz intensity`. Der Loader überspringt alles, was nicht genau `expected_fields` Zahlenspalten hat, und nimmt standardmäßig die Spalten 0–2 als X/Y/Z.

**Ziel u. a.:** prüfen, ob eine Datei eine zweite Fläche / einen Fremdblock enthält (z. B. Punkte weit weg von der Hauptplatte), und ob das an der Spaltenzuordnung/am Format liegt.

In [ ]:
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# Repo-Wurzel finden (funktioniert egal, wo der Kernel startet)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').exists()), Path.cwd())
SCAN_DIR = ROOT / 'data' / 'raw' / 'real_testscans'
print('Repo-Wurzel:', ROOT)
print('Scan-Ordner:', SCAN_DIR)

In [ ]:
def load_xyz(path, xyz_cols=(0, 1, 2), expected_fields=7, return_extra=False):
    """X/Y/Z aus einer Scanner-XYZ lesen.

    Zeilen ohne genau `expected_fields` Zahlenspalten (Kopfzeilen wie 'L####…')
    werden übersprungen. `xyz_cols` = welche Spalten X/Y/Z sind.
    Mit return_extra=True kommt zusätzlich die 7. Spalte (Intensität/Qualität)
    und die Dateizeilennummer je Punkt zurück (nützlich, um Blöcke zu finden).
    """
    path = Path(path)
    pts, extra, lineno = [], [], []
    n_skipped = 0
    with open(path, encoding='latin-1') as fh:
        for i, line in enumerate(fh):
            p = line.split()
            if len(p) != expected_fields:
                n_skipped += 1
                continue
            try:
                pts.append([float(p[xyz_cols[0]]), float(p[xyz_cols[1]]), float(p[xyz_cols[2]])])
                extra.append(float(p[6]) if expected_fields >= 7 else np.nan)
                lineno.append(i)
            except (ValueError, IndexError):
                n_skipped += 1
    pts = np.asarray(pts, float)
    print(f'{path.name}: {len(pts)} Punkte, {n_skipped} übersprungene Zeilen (Kopf/ungültig)')
    if len(pts):
        print(f'  BBox min {np.round(pts.min(0), 2)}  max {np.round(pts.max(0), 2)}')
    if return_extra:
        return pts, np.asarray(extra), np.asarray(lineno)
    return pts

In [ ]:
# Verfügbare Dateien auflisten
files = sorted(p for p in SCAN_DIR.iterdir() if p.suffix.lower() == '.xyz')
for i, f in enumerate(files):
    print(i, '·', f.name)

In [ ]:
# Datei wählen (Index aus der Liste oben) und laden
FILE_INDEX = 0
pts, extra, lineno = load_xyz(files[FILE_INDEX], xyz_cols=(0, 1, 2), return_extra=True)

In [ ]:
def show3d(pts, color=None, max_points=60000, title=''):
    """Interaktive 3D-Ansicht (drehbar). Bei vielen Punkten wird subsampled."""
    rng = np.random.default_rng(0)
    idx = rng.choice(len(pts), min(max_points, len(pts)), replace=False) if len(pts) > max_points else np.arange(len(pts))
    p = pts[idx]
    c = (p[:, 2] if color is None else np.asarray(color)[idx])
    fig = go.Figure(go.Scatter3d(
        x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers',
        marker=dict(size=1.5, color=c, colorscale='Viridis', colorbar=dict(title='Z (mm)')),
        hovertemplate='X %{x:.1f} · Y %{y:.1f} · Z %{z:.1f}<extra></extra>'))
    fig.update_layout(scene=dict(aspectmode='data'), height=700,
                      margin=dict(l=0, r=0, t=30, b=0), title=title)
    fig.show()

show3d(pts, title=files[FILE_INDEX].name)

In [ ]:
def show_projections(pts):
    """Drei orthografische Ansichten + Z-Histogramm — zeigt Nebenflächen sofort."""
    fig, ax = plt.subplots(1, 4, figsize=(18, 4.2))
    for a, (i, j, la, lb) in zip(ax[:3], [(0, 1, 'X', 'Y'), (0, 2, 'X', 'Z'), (1, 2, 'Y', 'Z')]):
        a.scatter(pts[:, i], pts[:, j], s=1, c='#444')
        a.set_xlabel(la + ' (mm)'); a.set_ylabel(lb + ' (mm)')
        a.set_title(f'{la}-{lb}'); a.set_aspect('equal', 'datalim')
    ax[3].hist(pts[:, 2], bins=120, color='#1976d2')
    ax[3].set_xlabel('Z (mm)'); ax[3].set_ylabel('Anzahl'); ax[3].set_title('Z-Histogramm')
    plt.tight_layout(); plt.show()

show_projections(pts)

## Fremdblöcke aufspüren

Falls eine zweite Fläche auftaucht: mit welchem Dateizeilen-Bereich hängt sie zusammen? Das zeigt, ob es ein zusammenhängender Abschnitt am Dateianfang/-ende ist (Format/Section) oder verstreut (echte Ausreißer).

In [ ]:
# Z-Schwelle: Punkte weit unter der Platte markieren und ihre Dateiposition zeigen
Z_THRESHOLD = -10.0
far = pts[:, 2] < Z_THRESHOLD
print(f'Punkte mit Z < {Z_THRESHOLD}: {far.sum()} von {len(pts)}')
if far.any():
    print(f'  Dateizeilen dieser Punkte: {lineno[far].min()} … {lineno[far].max()}')
    print(f'  Dateizeilen der übrigen:   {lineno[~far].min()} … {lineno[~far].max()}')
    print('  -> zusammenhängender Block?' , (lineno[far].max() < lineno[~far].min()) or (lineno[far].min() > lineno[~far].max()))

In [ ]:
# Optional: nur die Hauptplatte behalten (größte Ebene per RANSAC), Rest ausblenden
import open3d as o3d

def keep_main_plane(pts, plane_dist=1.0, groove_depth=8.0):
    """Behält Punkte nahe der größten Ebene plus das Nutband darunter."""
    pc = o3d.geometry.PointCloud(); pc.points = o3d.utility.Vector3dVector(pts)
    plane, _ = pc.segment_plane(0.5, 3, 1000)
    nrm = np.array(plane[:3]); L = np.linalg.norm(nrm)
    signed = (pts @ nrm + plane[3]) / L
    # Vorzeichen so, dass 'unten' (Nut) negativ ist: an der Punktmasse orientieren
    if np.median(signed) > 0:
        signed = -signed
    mask = (signed < plane_dist) & (signed > -groove_depth)
    print(f'behalten {mask.sum()} von {len(pts)}')
    return mask

mask = keep_main_plane(pts)
show3d(pts[mask], title='nur Hauptplatte')
show_projections(pts[mask])